# **Judicial Service Gen AI Chatbot For District Court**

---


## **FAQ LLM Pipeline (CSV → Embeddings → FAISS → Deterministic / RAG / Fallback)**

This notebook lets you build a **professional, zero-license-cost** FAQ system from a CSV with exactly **three columns**: `category, question, answer`.

**Highlights**
- **Deterministic** answers from CSV when similarity is high (safest)
- **RAG** (optional): use **Ollama** to compose a one-paragraph answer from top contexts when similarity is medium
- **Fallback**: when similarity is too low → `"Maaf, saya tidak bisa membantu untuk pertanyaan Anda."`
- **Auto-categorization** of the user question (KNN-majority from nearest neighbors)
- **Auto-reindex** when CSV changes (timestamp-based)
- **Logging** & optional candidate-FAQ capture for new content

> Run cells **top to bottom**. Replace `CSV_PATH` with your actual CSV path.


---

## **Imports & Configuration**

---

In [1]:
# FAQ LLM core from CSV to FAISS with deterministic routing RAG and fallback
import os, json, time, hashlib
from pathlib import Path
from typing import List, Dict, Any
import requests
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
from pathlib import Path
from IPython.display import display

# Environment flags to prevent TensorFlow and JAX from loading when using Transformers
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("USE_JAX", "0")
os.environ.setdefault("USE_TORCH", "1")

# Configuration for input data and index storage
CSV_PATH = Path("Data/standar_layanan_combined.csv")  # CSV file must contain columns category question answer
INDEX_DIR = Path("index_store"); INDEX_DIR.mkdir(parents=True, exist_ok=True)

EMB_MODEL = "intfloat/multilingual-e5-small"  # Multilingual lightweight embedding model
TOP_K = 20  # Maximum number of retrieval results used per query

# Thresholds used for three lane routing logic
HIGH_THRESHOLD = 0.880  # High confidence branch that answers directly from CSV
LOW_THRESHOLD  = 0.820  # Low confidence boundary that triggers fallback branch

# RAG configuration used when Ollama and the chosen model are available
ENABLE_RAG = True
OLLAMA_MODEL = "qwen2.5:7b(:q4_K_M)"
OLLAMA_HOST = "http://localhost:11434"

# Paths for logging user queries and storing candidate FAQ records
LOG_PATH = INDEX_DIR / "query_log.jsonl"
CANDIDATE_FAQ_PATH = INDEX_DIR / "candidate_faq.csv"


---

## **Utility Functions**

---

In [2]:
def read_csv_clean(csv_path: str) -> pd.DataFrame:
    # Load raw CSV into a DataFrame
    df = pd.read_csv(csv_path)
    # Validate required columns are present in the dataset
    required = {"category","question","answer"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"CSV must contain columns: {required}. Got: {df.columns.tolist()}")
    # Basic cleanup for null and whitespace only values
    df = df.dropna(subset=["category","question","answer"]).reset_index(drop=True)
    df["category"] = df["category"].astype(str).str.strip()
    df["question"] = df["question"].astype(str).str.strip()
    df["answer"]   = df["answer"].astype(str).str.strip()
    # Build doc_text field used as retrieval unit in the index
    df["doc_text"] = "[" + df["category"] + "] Q: " + df["question"] + "\nA: " + df["answer"]
    return df


def row_hash(cat: str, q: str, a: str) -> str:
    # Create a stable fingerprint for one FAQ row based on category question and answer
    return hashlib.sha1(f"{cat}||{q}||{a}".encode("utf-8")).hexdigest()


def load_embedder(name: str = EMB_MODEL):
    # Maintain a single shared embedding model instance during process lifetime
    if not hasattr(load_embedder, "_model"):
        load_embedder._model = SentenceTransformer(name)
    return load_embedder._model


def encode_passages(texts: List[str]) -> np.ndarray:
    # Follow E5 pattern by prefixing passage token before document texts
    model = load_embedder()
    return model.encode(
        ["passage: " + t for t in texts],
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)


def encode_queries(queries: List[str]) -> np.ndarray:
    # Follow E5 pattern by prefixing query token before user queries
    model = load_embedder()
    return model.encode(
        ["query: " + q for q in queries],
        normalize_embeddings=True,
    ).astype(np.float32)


def save_index(index, df: pd.DataFrame, csv_path: str):
    # Persist FAISS index on disk
    faiss.write_index(index, str(INDEX_DIR / "faq.index"))
    # Persist mapping frame that links index positions to original rows
    df.to_parquet(INDEX_DIR / "faq_mapping.parquet", index=False)
    # Store lightweight metadata for freshness checks and debugging
    meta = {
        "ts": time.time(),
        "rows": len(df),
        "csv_path": str(csv_path),
        "csv_mtime": os.path.getmtime(csv_path),
        "emb_model": EMB_MODEL,
    }
    with open(INDEX_DIR / "meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)


# Runtime cache for keeping DataFrame index and metadata in memory
_RUNTIME = {}


def load_index():
    # Load index mapping and metadata from the index directory
    index = faiss.read_index(str(INDEX_DIR / "faq.index"))
    df = pd.read_parquet(INDEX_DIR / "faq_mapping.parquet")
    with open(INDEX_DIR / "meta.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
    return df, index, meta


def get_store():
    # Lazily load and cache store objects for reuse across function calls
    if "df" not in _RUNTIME:
        df, index, meta = load_index()   # Call ensure_index_current at startup for safety
        _RUNTIME["df"], _RUNTIME["index"], _RUNTIME["meta"] = df, index, meta
    return _RUNTIME["df"], _RUNTIME["index"], _RUNTIME["meta"]


def index_exists() -> bool:
    # Check for presence of both index structure and mapping frame on disk
    return (INDEX_DIR / "faq.index").exists() and (INDEX_DIR / "faq_mapping.parquet").exists()


def is_index_outdated(csv_path: str) -> bool:
    # Compare CSV modification time with metadata to detect stale index
    meta_path = INDEX_DIR / "meta.json"
    if not meta_path.exists():
        return True
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    return meta.get("csv_mtime", 0) < os.path.getmtime(csv_path)

---

## **Build (or Rebuild) FAISS Index**

---

In [3]:
def build_index(csv_path: str = CSV_PATH):
    # Load cleaned FAQ data from the CSV file
    print(f"Loading CSV: {csv_path}")
    df = read_csv_clean(csv_path)

    # Encode all FAQ documents into dense vector representations
    print("Encoding passages (this may take a minute on first run)…")
    emb = encode_passages(df["doc_text"].tolist())

    # Build a FAISS index using inner product for similarity search
    print("Building FAISS index…")
    index = faiss.IndexFlatIP(emb.shape[1])   # cosine if normalized embeddings
    index.add(emb)

    # Persist index and mapping information on disk
    save_index(index, df, csv_path)
    print(f"Index saved to: {INDEX_DIR}")
    return df, index


# Build or rebuild index on module import when missing or outdated
if not index_exists() or is_index_outdated(CSV_PATH):
    _df, _index = build_index(CSV_PATH)
else:
    print("Existing index is up-to-date.")


Existing index is up-to-date.


---

## **Retrieval & Routing (Deterministic → RAG → Fallback)**

---

In [4]:
def retrieve(query: str, k: int = TOP_K) -> List[Dict[str, Any]]:
    # Retrieve top matching FAQ entries for a user query from the FAISS index
    df, index, meta = get_store()
    qv = encode_queries([query])
    D, I = index.search(qv, k)
    hits = []
    for score, idx in zip(D[0], I[0]):
        row = df.iloc[int(idx)]
        hits.append({
            "score": float(score),
            "category": row["category"],
            "question": row["question"],
            "answer": row["answer"],
            "doc_text": row["doc_text"]
        })
    return hits


def deterministic_answer(hit: Dict[str, Any]) -> str:
    # Produce a clean answer string from a single FAQ hit
    base = hit["answer"].strip()
    if not base.endswith((".", "!", "?")):
        base += "."
    return base.strip()


def predict_category_knn(hits: List[Dict[str, Any]], k: int = 10) -> str:
    # Predict category using a simple majority vote over top hits
    pool = hits[:k]
    cats = [h["category"] for h in pool]
    if not cats:
        return ""
    return max(set(cats), key=cats.count)


def log_interaction(payload: Dict[str, Any], log_path: Path = Path(LOG_PATH)):
    # Append interaction details to a JSON lines log file
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")


def add_candidate_faq(
    category_suggested: str,
    question: str,
    answer_draft: str = "",
    source: str = "user_query_fallback",
    status: str = "pending",
):
    # Store user question as a candidate FAQ for later curation
    rec = {
        "category_suggested": (category_suggested or "").strip(),
        "question": str(question).strip(),
        "answer_draft": (answer_draft or "").strip(),
        "source": source,
        "status": status,
        "ts": int(time.time()),
        "ts_iso": pd.Timestamp.utcnow().isoformat()
    }
    new_df = pd.DataFrame([rec])

    p = Path(CANDIDATE_FAQ_PATH)
    if p.exists():
        # Load existing candidate FAQ entries and normalize timestamp field
        old = pd.read_csv(p, keep_default_na=False)
        if "ts" in old.columns:
            ts_num = pd.to_numeric(old["ts"], errors="coerce")
            ts_sec = np.where(ts_num > 1e12, ts_num / 1000.0, ts_num)
            old["ts"] = pd.Series(ts_sec).round().astype("Int64")
        out = pd.concat([old, new_df], ignore_index=True)
        out = out.drop_duplicates(subset=["question", "category_suggested"], keep="last")
        out = out.sort_values(by="ts", kind="stable", na_position="first")
    else:
        # First candidate FAQ file creation path
        out = new_df

    out.to_csv(p, index=False)


def call_ollama(
    prompt: str,
    model: str = OLLAMA_MODEL,
    host: str = OLLAMA_HOST,
    timeout: int = 120,
) -> str:
    # Quick guard so no empty prompt is sent to Ollama
    if not prompt or not str(prompt).strip():
        return "(Prompt ke Ollama kosong. Cek make_prompt/contexts.)"

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0, "num_ctx": 2048}
    }
    try:
        # Call Ollama HTTP endpoint and return plain text response
        r = requests.post(f"{host}/api/generate", json=payload, timeout=timeout)
        if r.status_code == 200:
            txt = (r.json().get("response") or "").strip()
            if not txt:
                return "(Ollama mengembalikan respons kosong.)"
            return txt
        return f"(Ollama error: HTTP {r.status_code} - {r.text[:300]})"
    except Exception as e:
        # Return structured error text when Ollama is not reachable
        return f"(Ollama not reachable: {e})"


def make_prompt(user_q, contexts):
    # Build a compact RAG prompt using top FAQ contexts
    def clip(t, n=600):
        return t[:n]
    ctx_text = "\n\n---\n".join([clip(c["doc_text"]) for c in contexts[:2]])
    return (
        "Anda adalah asisten FAQ pengadilan. "
        "Jawab ringkas (maks 1 paragraf) dalam bahasa Indonesia. "
        "Dasarkan jawaban HANYA pada KONTEKS di bawah. "
    )


def answer_question(q: str) -> Dict[str, Any]:
    # Main routing logic that decides between deterministic answer RAG or fallback
    hits = retrieve(q, k=TOP_K)
    if not hits:
        msg = "Maaf, saya tidak bisa membantu untuk pertanyaan Anda."
        log_interaction({"ts": time.time(), "question": q, "mode": "NO_HITS", "answer": msg})
        return {"mode": "NO_HITS", "answer": msg, "top_score": 0.0, "predicted_category": ""}

    top = hits[0]
    s = float(top["score"])
    pred_cat = predict_category_knn(hits)

    # High confidence route that uses direct FAQ answer
    if s >= HIGH_THRESHOLD:
        out = deterministic_answer(top)
        log_interaction({
            "ts": time.time(),
            "question": q,
            "mode": "DETERMINISTIC",
            "top_score": s,
            "predicted_category": pred_cat,
            "used_categories": [h["category"] for h in hits[:3]],
        })
        return {"mode": "DETERMINISTIC", "answer": out, "top_score": s, "predicted_category": pred_cat}

    # Mid confidence route that tries RAG when enabled
    elif s >= LOW_THRESHOLD:
        if ENABLE_RAG:
            ctx = hits[:3]
            prompt = make_prompt(q, ctx)
            out = call_ollama(prompt)
            if not out or not out.strip() or out.startswith("("):
                # Lightweight fallback that exposes a few similar questions
                suggest = "\n".join(
                    [f"- [{h['category']}] {h['question']} (skor {h['score']:.3f})" for h in ctx]
                )
                out = (
                    "Saya belum bisa merangkum jawaban dari konteks.\n"
                    "Pertanyaan Anda mirip dengan:\n" + suggest
                )
            log_interaction({
                "ts": time.time(),
                "question": q,
                "mode": "RAG",
                "top_score": s,
                "predicted_category": pred_cat,
                "used_categories": [h["category"] for h in ctx],
            })
            return {"mode": "RAG", "answer": out, "top_score": s, "predicted_category": pred_cat}

    # Low confidence route that logs and proposes candidate FAQ
    else:
        msg = "Maaf, saya tidak bisa membantu untuk pertanyaan Anda."
        log_interaction({
            "ts": time.time(),
            "question": q,
            "mode": "FALLBACK",
            "top_score": s,
            "predicted_category": pred_cat,
            "used_categories": [],
        })
        add_candidate_faq(pred_cat, q, source="user_query_fallback", status="pending")
        return {"mode": "FALLBACK", "answer": msg, "top_score": s, "predicted_category": pred_cat}

---

## **Question-Answer Testing**

---

In [5]:
# Example query to test the FAQ assistant workflow
question = "Bagaimana mendaftar perkara via e-Court?"
answer_question(question)

{'mode': 'DETERMINISTIC',
 'answer': 'Kelengkapan meliputi Advokat terverifikasi/tervalidasi Pengadilan Tinggi dapat langsung mendaftar via e-Court (ecourt.mahkamahagung.go.id), Pengguna Lain membuat akun e-Court melalui PN dengan syarat email aktif, fotokopi KTP, nama bank, nomor rekening, dan nomor handphone; setelah akun aktif dapat mendaftar perkara melalui e-Court. Penataan sejak awal membantu verifikasi cepat, pencatatan register, dan kelancaran proses berikutnya tanpa pengembalian berkas hingga pengesahan pejabat yang berwenang.',
 'top_score': 0.8941890001296997,
 'predicted_category': 'STANDAR PELAYANAN PENERIMAAN PERKARA MELALUI E-COURT'}

In [6]:
# Example query to test the FAQ assistant workflow
question = "Kapan Ketua baru dilantik?"
answer_question(question)


{'mode': 'FALLBACK',
 'answer': 'Maaf, saya tidak bisa membantu untuk pertanyaan Anda.',
 'top_score': 0.7643812894821167,
 'predicted_category': 'STANDAR PELAYANAN KEBERATAN ATAS PUTUSAN GUGATAN SEDERHANA'}

In [7]:
# Example query to test the FAQ assistant workflow
question = "Siapa itu Joko Widodo?"
answer_question(question)

{'mode': 'FALLBACK',
 'answer': 'Maaf, saya tidak bisa membantu untuk pertanyaan Anda.',
 'top_score': 0.7470119595527649,
 'predicted_category': 'informasi umum'}

In [8]:
# Example query to test the FAQ assistant workflow
question = "Apa itu PTSP?"
answer_question(question)

{'mode': 'DETERMINISTIC',
 'answer': 'Pelayanan Terpadu Satu Pintu menyediakan sarana yang memudahkan masyarakat mengurus urusan peradilan mulai dari pendaftaran pemantauan perkembangan hingga pembayaran sehingga proses terasa tertib mudah dan hemat waktu. Penjelasan singkat ini diharapkan membantu memahami konteksnya saat berhadapan dengan layanan di meja informasi maupun ruang sidang.',
 'top_score': 0.8869751691818237,
 'predicted_category': 'informasi umum'}

In [9]:
# Example query to test the FAQ assistant workflow
question = "Jam berapa pelayanan PTSP?"
answer_question(question)

{'mode': 'RAG',
 'answer': 'Saya belum bisa merangkum jawaban dari konteks.\nPertanyaan Anda mirip dengan:\n- [informasi umum] Sebutkan pengertian PTSP pada praktik peradilan umum (skor 0.871)\n- [informasi umum] Sebutkan pengertian PTSP pada praktik peradilan umum (skor 0.871)\n- [informasi umum] Sebutkan pengertian PTSP pada praktik peradilan umum (skor 0.871)',
 'top_score': 0.8714950084686279,
 'predicted_category': 'informasi umum'}

In [10]:
# Example query to test the FAQ assistant workflow
question = "Apa saja yang tidak boleh dibawa ke ruang sidang?"
answer_question(question)

{'mode': 'RAG',
 'answer': 'Saya belum bisa merangkum jawaban dari konteks.\nPertanyaan Anda mirip dengan:\n- [informasi umum] Pada eksekusi putusan apa arti Ruang Tahanan Sementara (skor 0.841)\n- [informasi umum] Pada eksekusi putusan apa arti Tata Tertib Persidangan (skor 0.841)\n- [informasi umum] Pada tahap upaya hukum apa yang dimaksud Ruang Tahanan Sementara (skor 0.840)',
 'top_score': 0.8412904739379883,
 'predicted_category': 'informasi umum'}

In [11]:
# Example query to test the FAQ assistant workflow
question = "Bagaimana cara mengajukan banding atas putusan pengadilan?"
answer_question(question)

{'mode': 'DETERMINISTIC',
 'answer': 'Urutan kerja diterapkan berkesinambungan: Pemohon menyatakan hendak mencabut permohonan banding dan menunjukkan salinan putusan yang diajukan banding; PTSP membuat akta pencabutan permohonan banding; Tanda tangan Panitera dimintakan dan akta diserahkan kepada Pemohon; Data pencabutan diinput ke SIPP. Rangkaian ini memastikan keterlacakan administrasi, kejelasan peran, serta validitas keluaran hingga siap dimanfaatkan oleh pemohon atau aparat penegak hukum yang berkepentingan.',
 'top_score': 0.8823426365852356,
 'predicted_category': 'STANDAR PELAYANAN PENERIMAAN PERMOHONAN PENCABUTAN UPAYA HUKUM BANDING'}

In [12]:
# Example query to test the FAQ assistant workflow
question = "Apakah boleh mancing di ruang sidang?"
answer_question(question)

{'mode': 'RAG',
 'answer': 'Saya belum bisa merangkum jawaban dari konteks.\nPertanyaan Anda mirip dengan:\n- [informasi umum] Bagaimana peranan Tata Tertib Persidangan dalam administrasi perkara (skor 0.840)\n- [informasi umum] Pada transparansi informasi publik apa maksud Ruang Tahanan Sementara (skor 0.836)\n- [informasi umum] Bagaimana Tata Tertib Persidangan membantu akses layanan peradilan (skor 0.835)',
 'top_score': 0.8398040533065796,
 'predicted_category': 'informasi umum'}

In [13]:
# Example query to test the FAQ assistant workflow
question = "Bagaimana membuat akun e-court?"
answer_question(question)

{'mode': 'DETERMINISTIC',
 'answer': 'Kelengkapan meliputi Advokat terverifikasi/tervalidasi Pengadilan Tinggi dapat langsung mendaftar via e-Court (ecourt.mahkamahagung.go.id), Pengguna Lain membuat akun e-Court melalui PN dengan syarat email aktif, fotokopi KTP, nama bank, nomor rekening, dan nomor handphone; setelah akun aktif dapat mendaftar perkara melalui e-Court. Penataan sejak awal membantu verifikasi cepat, pencatatan register, dan kelancaran proses berikutnya tanpa pengembalian berkas hingga pengesahan pejabat yang berwenang.',
 'top_score': 0.8929436802864075,
 'predicted_category': 'STANDAR PELAYANAN PENERIMAAN PERKARA MELALUI E-COURT'}

In [14]:
# Example query to test the FAQ assistant workflow
question = "Bagaimana cara izin besuk?"
answer_question(question)

{'mode': 'RAG',
 'answer': 'Saya belum bisa merangkum jawaban dari konteks.\nPertanyaan Anda mirip dengan:\n- [STANDAR PELAYANAN PENERIMAAN PERMOHONAN IZIN BESUK TAHANAN HAKIM] Berkaitan dengan tata cara standar pelayanan penerimaan permohonan izin besuk tahanan hakim, bagaimana urutan langkah dari penerimaan sampai keluaran diserahkan? (skor 0.871)\n- [STANDAR PELAYANAN PENERIMAAN PERMOHONAN IZIN BESUK TAHANAN HAKIM] Menurut SOP yang berlaku untuk standar pelayanan penerimaan permohonan izin besuk tahanan hakim, bagaimana urutan langkah dari penerimaan sampai keluaran diserahkan? (skor 0.870)\n- [STANDAR PELAYANAN PENERIMAAN PERMOHONAN IZIN BESUK TAHANAN HAKIM] Berlandaskan pedoman standar pelayanan penerimaan permohonan izin besuk tahanan hakim, bagaimana urutan langkah dari penerimaan sampai keluaran diserahkan? (skor 0.870)',
 'top_score': 0.8711580038070679,
 'predicted_category': 'STANDAR PELAYANAN PENERIMAAN PERMOHONAN IZIN BESUK TAHANAN HAKIM'}

In [15]:
# Example query to test the FAQ assistant workflow
question = "Bagaimana cara meminta salinan putusan?"
answer_question(question)

{'mode': 'DETERMINISTIC',
 'answer': 'Tahapan layanan dijalankan berurutan: PTSP menerima permohonan salinan putusan; Staf menindaklanjuti, mencatat di register, mencari arsip dan menyalin berkas; PTSP memberi catatan dan paraf, serta memintakan tanda tangan Panitera; PTSP menyerahkan formulir biaya untuk pembayaran di kasir; PTSP menyerahkan salinan putusan kepada pemohon. Rangkaian ini memastikan keterlacakan administrasi, kejelasan peran, dan keluaran yang sah sebelum dikembalikan kepada pemohon.',
 'top_score': 0.9022705554962158,
 'predicted_category': 'STANDAR PELAYANAN PENDAFTARAN PERMOHONAN FOTOKOPI SALINAN PUTUSAN PUTUSAN PENGADILAN'}

In [16]:
# Example query to test the FAQ assistant workflow
question = "Bagaimana cara meminta limpahan berkas?"
answer_question(question)

{'mode': 'RAG',
 'answer': 'Saya belum bisa merangkum jawaban dari konteks.\nPertanyaan Anda mirip dengan:\n- [STANDAR PELAYANAN PENERIMAAN PERKARA PELANGGARAN LALU LINTAS] Pada ketentuan resmi standar pelayanan penerimaan perkara pelanggaran lalu lintas, berkas apa yang menjadi prasyarat agar permohonan diterima? (skor 0.865)\n- [STANDAR PELAYANAN PENERIMAAN PERKARA PELANGGARAN LALU LINTAS] Jika merujuk prosedur standar pelayanan penerimaan perkara pelanggaran lalu lintas, berkas apa yang menjadi prasyarat agar permohonan diterima? (skor 0.864)\n- [STANDAR PELAYANAN PENERIMAAN PERKARA PELANGGARAN LALU LINTAS] Dalam standar pelayanan standar pelayanan penerimaan perkara pelanggaran lalu lintas, berkas apa yang menjadi prasyarat agar permohonan diterima? (skor 0.864)',
 'top_score': 0.8647624254226685,
 'predicted_category': 'STANDAR PELAYANAN PENERIMAAN PERKARA PELANGGARAN LALU LINTAS'}

In [17]:
# Example query to test the FAQ assistant workflow
question = "Bagaimana cara meminta penyitaan?"
answer_question(question)

{'mode': 'RAG',
 'answer': 'Saya belum bisa merangkum jawaban dari konteks.\nPertanyaan Anda mirip dengan:\n- [STANDAR PELAYANAN PENERIMAAN PERMOHONAN IZIN/ PERSETUJUAN PENYITAAN ATAU PENGGELEDAHAN] Berkaitan dengan tata cara standar pelayanan penerimaan permohonan izin/ persetujuan penyitaan atau penggeledahan, apa saja yang perlu disertakan supaya verifikasi berjalan tanpa hambatan? (skor 0.866)\n- [STANDAR PELAYANAN PENERIMAAN PERMOHONAN IZIN/ PERSETUJUAN PENYITAAN ATAU PENGGELEDAHAN] Menurut SOP yang berlaku untuk standar pelayanan penerimaan permohonan izin/ persetujuan penyitaan atau penggeledahan, apa saja yang perlu disertakan supaya verifikasi berjalan tanpa hambatan? (skor 0.864)\n- [STANDAR PELAYANAN PENERIMAAN PERMOHONAN IZIN/ PERSETUJUAN PENYITAAN ATAU PENGGELEDAHAN] Menurut uraian proses standar pelayanan penerimaan permohonan izin/ persetujuan penyitaan atau penggeledahan, apa saja yang perlu disertakan supaya verifikasi berjalan tanpa hambatan? (skor 0.863)',
 'top_scor

In [18]:
# Example query to test the FAQ assistant workflow
question = "Bagaimana Syarat mengajukan gugatan?"
answer_question(question)

{'mode': 'RAG',
 'answer': 'Saya belum bisa merangkum jawaban dari konteks.\nPertanyaan Anda mirip dengan:\n- [STANDAR PELAYANAN KEBERATAN ATAS PUTUSAN GUGATAN SEDERHANA] Saat mengajukan keberatan atas putusan gugatan sederhana di pengadilan, kelengkapan apa yang harus dibawa saat pendaftaran di loket? (skor 0.873)\n- [STANDAR PELAYANAN PERKARA GUGATAN SEDERHANA] Saat mengajukan gugatan sederhana perdata di pengadilan, apa daftar berkas yang menjadi prasyarat sebelum pemeriksaan awal dilakukan? (skor 0.872)\n- [STANDAR PELAYANAN KEBERATAN ATAS PUTUSAN GUGATAN SEDERHANA] Saat mengajukan keberatan atas putusan gugatan sederhana di pengadilan, apa daftar berkas yang menjadi prasyarat sebelum pemeriksaan awal dilakukan? (skor 0.872)',
 'top_score': 0.8730061054229736,
 'predicted_category': 'STANDAR PELAYANAN KEBERATAN ATAS PUTUSAN GUGATAN SEDERHANA'}

In [19]:
# Example query to test the FAQ assistant workflow
question = "Bagaimana cara membayar biaya perkara?"
answer_question(question)

{'mode': 'DETERMINISTIC',
 'answer': 'Uang muka biaya perkara yang dihitung berdasarkan komponen tertentu menyediakan sarana yang memudahkan masyarakat mengurus urusan peradilan mulai dari pendaftaran pemantauan perkembangan hingga pembayaran sehingga proses terasa tertib mudah dan hemat waktu. Penjelasan singkat ini diharapkan membantu memahami konteksnya saat berhadapan dengan layanan di meja informasi maupun ruang sidang.',
 'top_score': 0.8845093250274658,
 'predicted_category': 'informasi umum'}

In [20]:
# Example query to test the FAQ assistant workflow
question = "Bagaimana cara membayar biaya panjar?"
answer_question(question)

{'mode': 'RAG',
 'answer': 'Saya belum bisa merangkum jawaban dari konteks.\nPertanyaan Anda mirip dengan:\n- [informasi umum] Sebutkan pengertian Biaya Panjar pada praktik peradilan umum (skor 0.880)\n- [informasi umum] Sebutkan pengertian Biaya Panjar pada praktik peradilan umum (skor 0.880)\n- [informasi umum] Sebutkan pengertian Biaya Panjar pada praktik peradilan umum (skor 0.880)',
 'top_score': 0.8797845840454102,
 'predicted_category': 'informasi umum'}

In [21]:
# Example query to test the FAQ assistant workflow
question = "Apa itu panjar biaya perkara?"
answer_question(question)

{'mode': 'DETERMINISTIC',
 'answer': 'Uang muka biaya perkara yang dihitung berdasarkan komponen tertentu menyediakan sarana yang memudahkan masyarakat mengurus urusan peradilan mulai dari pendaftaran pemantauan perkembangan hingga pembayaran sehingga proses terasa tertib mudah dan hemat waktu. Penjelasan singkat ini diharapkan membantu memahami konteksnya saat berhadapan dengan layanan di meja informasi maupun ruang sidang.',
 'top_score': 0.9046648740768433,
 'predicted_category': 'informasi umum'}

---

## **Auto-Reindex When CSV Changes**

---

In [22]:
def ensure_index_current(csv_path: str = CSV_PATH):
    # Rebuild index when missing or when source CSV has changed
    if not index_exists() or is_index_outdated(csv_path):
        print("CSV changed or index missing — rebuilding…")
        build_index(csv_path)
    else:
        # Skip rebuild when existing index is still valid
        print("Index is up-to-date.")

# Run at startup so retrieval always uses the latest index
ensure_index_current(CSV_PATH)

Index is up-to-date.


---

## **Batch Queries (Optional)**

---

In [23]:
# Sample user questions covering several service topics
questions = [
    "Bagaimana mendaftar perkara via e-Court?",
    "Apa itu panjar biaya perkara?",
    "Kapan layanan praperadilan dibuka?"
    "Bagaimana cara mengajukan banding atas putusan pengadilan?",
    "Apa syarat untuk mendapatkan bantuan hukum gratis?",
    "Dimana saya bisa menemukan jadwal sidang terbaru?"
    "Kapan Ketua baru dilantik?"
]

# Container for structured results per question
rows = []
for q in questions:
    # Invoke core question answering function and capture the response
    res = answer_question(q)
    rows.append({
        "question": q,
        "answer": res["answer"],
        "mode": res["mode"],
        "top_score": res["top_score"],
        "predicted_category": res["predicted_category"]
    })

# Build a DataFrame for quick inspection of outcomes
pd.DataFrame(rows)

,question,answer,mode,top_score,predicted_category
0,Bagaimana mendaftar perkara via e-Court?,Kelengkapan meliputi Advokat terverifikasi/ter...,DETERMINISTIC,0.894189,STANDAR PELAYANAN PENERIMAAN PERKARA MELALUI E...
1,Apa itu panjar biaya perkara?,Uang muka biaya perkara yang dihitung berdasar...,DETERMINISTIC,0.904665,informasi umum
2,Kapan layanan praperadilan dibuka?Bagaimana ca...,Saya belum bisa merangkum jawaban dari konteks...,RAG,0.879501,STANDAR PELAYANAN PENERIMAAN PERMOHONAN UPAYA ...
3,Apa syarat untuk mendapatkan bantuan hukum gra...,Saya belum bisa merangkum jawaban dari konteks...,RAG,0.855460,informasi umum
4,Dimana saya bisa menemukan jadwal sidang terba...,Saya belum bisa merangkum jawaban dari konteks...,RAG,0.848873,informasi umum


---

## **Candidate FAQ Review**

---

In [24]:
SOURCE_DESC = {
    "user_query_fallback": "Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).",
    "rag_summary": "Otomatis: ringkasan RAG disimpan sebagai draf untuk ditinjau.",
    "user_log": "Dimasukkan dari log/monitoring manual.",
    "manual": "Ditambahkan manual oleh admin."
}

STATUS_DESC = {
    "pending": "Belum ditinjau kurator.",
    "drafted": "Sudah ada draf jawaban, menunggu review.",
    "approved": "Sudah disetujui & diintegrasikan ke CSV utama.",
    "rejected": "Tidak relevan/ditolak."
}

path = Path(CANDIDATE_FAQ_PATH)

if not path.exists():
    print("No candidate_faq.csv yet.")
else:
    df = pd.read_csv(path, keep_default_na=False)

    # Convert numeric ts column into a human readable timestamp column
    if "ts" in df.columns:
        ts_num = pd.to_numeric(df["ts"], errors="coerce")
        ts_sec = np.where(ts_num > 1e12, ts_num / 1000.0, ts_num)
        ts_ser = pd.to_datetime(pd.Series(ts_sec), unit="s", utc=True)
        df["ts_readable"] = ts_ser.dt.tz_localize(None)

    # Add explanation columns for source and status fields
    df["source_desc"] = df["source"].map(SOURCE_DESC).fillna("Sumber lain/unknown")
    df["status_desc"] = df["status"].map(STATUS_DESC).fillna("Status lain/unknown")

    # Configure display options for wide preview in notebooks
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)

    display(df[[
        "category_suggested",
        "question",
        "answer_draft",
        "source", "source_desc",
        "status", "status_desc",
        "ts_readable"
    ]].tail(10))


,category_suggested,question,answer_draft,source,source_desc,status,status_desc,ts_readable
3,Upaya Hukum Banding (Pidana),Siapa Guntur Pambudi Wijaya?,,user_query_fallback,Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).,pending,Belum ditinjau kurator.,2025-10-30 15:20:34
4,Inzage (Pemeriksaan Berkas) Perkara Banding,Apakah boleh mancing di ruang sidang?,,user_query_fallback,Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).,pending,Belum ditinjau kurator.,2025-10-30 15:20:42
5,Standar Pelayanan Hukum - Pengaduan melalui Meja SIWAS,Siapa Guntur Pambudi Wijaya?,,user_query_fallback,Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).,pending,Belum ditinjau kurator.,2025-11-01 16:10:50
6,informasi umum,Kapan Ketua baru dilantik?,,user_query_fallback,Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).,pending,Belum ditinjau kurator.,2025-11-04 01:07:47
7,STANDAR PELAYANAN PENERIMAAN PERMOHONAN DIVERSI,Siapa Guntur Pambudi Wijaya?,,user_query_fallback,Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).,pending,Belum ditinjau kurator.,2025-11-04 01:07:49
8,informasi umum,Apa saja yang tidak boleh dibawa ke ruang sidang?,,user_query_fallback,Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).,pending,Belum ditinjau kurator.,2025-11-04 01:07:49
9,informasi umum,Siapa nama hakim ketua di Pengadilan Negeri Kotabaru?,,user_query_fallback,Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).,pending,Belum ditinjau kurator.,2025-11-11 06:13:37
10,STANDAR PELAYANAN PENERIMAAN PERMOHONAN UPAYA HUKUM BANDING,Siapa Guntur Pambudi Wijaya?,,user_query_fallback,Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).,pending,Belum ditinjau kurator.,2025-11-11 06:13:37
11,STANDAR PELAYANAN KEBERATAN ATAS PUTUSAN GUGATAN SEDERHANA,Kapan Ketua baru dilantik?,,user_query_fallback,Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).,pending,Belum ditinjau kurator.,2025-11-16 15:09:59
12,informasi umum,Siapa itu Joko Widodo?,,user_query_fallback,Otomatis: pertanyaan user jatuh ke fallback (skor < LOW_THRESHOLD).,pending,Belum ditinjau kurator.,2025-11-16 15:09:59
